In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 13 — PIPELINES: A ARQUITETURA DA CONFIABILIDADE

> "Um amador se preocupa com o algoritmo. Um profissional se preocupa com o fluxo de dados. O melhor modelo do mundo é inútil se os dados em produção não forem processados exatamente como no treinamento."
> — A Regra de Ouro da Implantação

Eu tinha vários modelos promissores e estava tentado a declarar vitória. Mas a memória do OncoClassify 1.0 me assombrava: aquele fracasso não foi só de algoritmo, foi de **processo**. Meu fluxo até aqui, embora cuidadoso, era uma sequência de passos manuais mantidos unidos pela minha disciplina. E a disciplina humana é falível. O que aconteceria se, em seis meses, eu retreinasse o modelo e esquecesse de aplicar a mesma normalização ao teste? Cada pequeno desvio poderia introduzir *data leakage* silencioso.

Eu não estava construindo um modelo, mas um sistema. E um sistema precisa de arquitetura. Precisava encapsular todo o fluxo — pré-processamento e modelagem — em um único objeto inviolável. Precisava de um **Pipeline**.

## Pipelines: Consistência e Prevenção de Erros

Um **Pipeline** encadeia passos de processamento e um estimador final num único objeto. Ao chamar `.fit()`, ele ajusta cada transformador e treina o modelo em sequência; ao chamar `.predict()`, aplica os **mesmos** parâmetros aprendidos no treino aos novos dados. Os benefícios:

**Previne *data leakage*** — na validação cruzada, o *scaler* e a seleção são reajustados **dentro de cada *fold***, usando só o treino daquele *fold*.

**Legibilidade e reprodutibilidade** — um objeto único representa todo o fluxo; pode ser salvo e recarregado em produção.

**Otimização conjunta** — permite ajustar hiperparâmetros do modelo *e* do pré-processamento de uma vez (Capítulo 14).

Vamos montar um *pipeline* completo com três passos: seleção de *features* (`SelectKBest`, do Capítulo 7), normalização e o classificador SVM.

#### Código 13.1: Um Pipeline Completo


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

def info_mutua(X, y):
    return mutual_info_classif(X, y, random_state=42)

pipeline = Pipeline([
    ("selecao", SelectKBest(score_func=info_mutua, k=15)),  # 15 melhores features
    ("escala",  StandardScaler()),
    ("svm",     SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)),
])

r = cross_validate(pipeline, X_train, y_train, cv=5,
                   scoring={"acuracia": "accuracy", "recall_Maligno": recall_maligno})
print(f"Acuracia media:       {r['test_acuracia'].mean():.4f}")
print(f"Recall Maligno medio: {r['test_recall_Maligno'].mean():.4f}")

# O pipeline inteiro age como um unico estimador. Treinamos no treino e
# demonstramos a previsao nas primeiras linhas do PROPRIO treino --
# o cofre (X_test) continua lacrado ate o Capitulo 24.
pipeline.fit(X_train, y_train)
print("Previsoes (treino, 8 primeiras):", pipeline.predict(X_train)[:8].tolist())

Acuracia media:       0.9417
Recall Maligno medio: 0.8867
Previsoes (treino, 8 primeiras): [0, 1, 0, 1, 0, 1, 0, 1]


> **⚠️ ATENÇÃO — O ganho do pipeline não é acurácia**
>
> Reparou? Com `k=15`, a acurácia **caiu** para 94,17% (contra 97,50% do SVM-RBF com todas as *features*). Cortar para 15 *features* foi agressivo demais — e é honesto dizer isso, em vez de fingir que "ficou quase igual". O que ganhamos aqui **não é desempenho, é robustez**: um único objeto, à prova de erro humano, que reaplica exatamente o mesmo processamento em qualquer dado futuro. O *quanto* cortar (o valor de `k`) é um hiperparâmetro — e no próximo capítulo deixaremos a máquina descobrir o `k` ideal, em vez de chutarmos 15.

A robustez é o ponto. Todo o processo — selecionar, escalar, treinar — está encapsulado. Quando um dia chamarmos `.predict()` em dados novos, o *pipeline* garantirá que eles passem pelos mesmos passos, com os parâmetros aprendidos **apenas** no treino — mas isso fica para o Capítulo 24, quando finalmente abriremos o cofre. O risco de erro humano caiu drasticamente.

> **📌 CHECKLIST DO CAPÍTULO 13**
>
> ✅ Entende o que é um Pipeline e por que ele é a espinha dorsal de um projeto robusto.
>
> ✅ Sabe que ele previne *leakage*, melhora legibilidade e permite otimização conjunta.
>
> ✅ Executou o Código 13.1 e observou que `k=15` **reduziu** a acurácia — o pipeline entrega robustez, não desempenho por si só.
>
> ✅ Reconhece que `k` é um hiperparâmetro a ser otimizado, não chutado.
>
> **⚠️ CRÍTICO:** A partir daqui, pense em *pipelines*, não em modelos isolados. Todo fluxo deve ser encapsulado para ser reproduzível e pronto para produção.

Eu finalmente tinha minha arquitetura. O *pipeline* não era só código: era uma apólice de seguro contra meus próprios erros futuros. Mas seus parâmetros — `k=15`, `C=1`, `gamma='scale'` — eram suposições razoáveis, nada mais. Como saber se `k=25` ou `C=10` não desbloqueariam um desempenho maior? Fazer isso à mão seria interminável. Eu precisava automatizar a busca. Era hora de caçar os hiperparâmetros.
